In [ ]:
import itertools as it
from collections import deque
import copy
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict


from collections import defaultdict
from matplotlib.ticker import MaxNLocator


from sklearn.preprocessing import StandardScaler
from river import stream

from sklearn.svm import OneClassSVM
from sklearn.inspection import DecisionBoundaryDisplay
from river import tree

# Import from our adazor package
from adazor.distributions import GaussianMixture, DriftingMixtureStream
from adazor.detectors import (
    NoDriftDetector,
    ThresholdDriftDetector,
    ZTestDriftDetector,
)
from adazor.models import (
    to_array,
    HoeffdingTreeClassifier,
    GaussianNBClassifier,
    PAClassifier,
    SklearnLinearSVCClassifier,
    SklearnRBFSVCClassifier,
    AdaptiveRandomForestClassifier,
    LogisticRegressionClassifier,
    SklearnMLPClassifier,
)


In [2]:
def load_xy_from_csv(
    csv_path: str,
    target_col: str = "target",
    standardize: bool = False,
    dtype=np.float32,
):
    """
    Loads a CSV with variable number of feature columns and a fixed target column name.
    Returns: X (np.ndarray), y (np.ndarray), feature_names (list[str])
    """
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(f"Missing '{target_col}' column. Found: {list(df.columns)}")

    y = df[target_col].to_numpy()
    feature_df = df.drop(columns=[target_col])
    feature_names = list(feature_df.columns)

    X = feature_df.to_numpy(dtype=dtype)

    if standardize:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

    return X, y, feature_names


In [3]:

class ArrayStream:
    """
    Stateful stream over fixed X, y.
    Emits feature dict keys as x_0, x_1, ..., x_{d-1} so ht2d.models.to_array works.
    """
    def __init__(self, X, y, drift_points=None):
        self.X = X
        self.y = y
        self.drift_points = drift_points

        d = X.shape[1]
        feature_names = [f"x_{i}" for i in range(d)]

        self._it = stream.iter_array(X, y, feature_names=feature_names)

    def sample(self):
        return self._it



In [4]:
def append_jsonl(path: str, record: dict):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    rec = dict(record)
    rec["timestamp_unix"] = time.time()

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")


In [6]:
keep_fraction = 0.1  # same as in the original notebook


def accumulate_examples(stream, n, x_window, y_window, keep_fraction=0.0):
    """Refill sliding windows from a stream, keeping a fraction of recent data."""
    if not (0.0 <= keep_fraction <= 1.0):
        raise ValueError("keep_fraction must be between 0.0 and 1.0")

    m = int(len(x_window) * keep_fraction)

    if m > 0:
        x_recent = list(it.islice(x_window, len(x_window) - m, None))
        y_recent = list(it.islice(y_window, len(y_window) - m, None))
    else:
        x_recent, y_recent = [], []

    x_window.clear()
    y_window.clear()
    x_window.extend(x_recent)
    y_window.extend(y_recent)

    n_new = n - len(x_window)
    for x, y in it.islice(stream.sample(), n_new):
        x_window.append(to_array(x))
        y_window.append(y)
###############################################

def generate_figure_all(accuracies, detected_drifts_list, drift_points=None, title=None, figsize=(8,3)):
    """
    New API matching run_all_modules_on_dataset:
      - accuracies: list of runs, each run is list of (t, acc, is_drift)
      - detected_drifts_list: list of runs, each run is list of (t_det, acc_at_t)
      - drift_points: optional list of ground-truth drift points or intervals
      - title: string for figure title
    Returns: matplotlib.figure.Figure
    """
    fig, ax = plt.subplots(figsize=figsize)
    num_drifts = []

    for acc_run, det_run in zip(accuracies, detected_drifts_list):
        num_drifts.append(len(det_run))
        # acc_run is list of (t, acc, is_drift)
        if len(acc_run) == 0:
            continue
        t_vals, acc_vals, _ = zip(*acc_run)
        ax.step(t_vals, acc_vals, color="blue", alpha=0.3)

        if det_run:
            t_det, acc_det = zip(*det_run)
            ax.scatter(t_det, acc_det, color="red", label="Detected drifts", s=10)

    # visualize true drift points (both gradual and sudden) if provided
    if drift_points:
        for drift in drift_points:
            if isinstance(drift, int):
                ax.axvline(drift, color="gray", linewidth=1.5, alpha=1.0)
            else:
                start_drift, end_drift = drift
                ax.axvspan(start_drift, end_drift, color="gray", alpha=0.2)
                ax.axvline(start_drift, color="gray", linewidth=1.5, alpha=1.0)
                ax.axvline(end_drift, color="gray", linewidth=1.5, alpha=1.0)

    # title with mean ± std of detected drifts
    if num_drifts:
        mean_drifts = float(np.mean(num_drifts))
        std_drifts = float(np.std(num_drifts, ddof=1)) if len(num_drifts) > 1 else 0.0
        msg = f"Detected drifts: {mean_drifts:.2f} ± {std_drifts:.2f} (n={len(num_drifts)})"
    else:
        msg = "Detected drifts: 0 (n=0)"

    if title:
        ax.set_title(f"{title} — {msg}")
    else:
        ax.set_title(msg)

    ax.set_xlabel("")  # no detector available in this API; set if you want
    ax.set_ylabel("accuracy")

    ax.set_xlim(left=0)
    ax.grid(alpha=0.2)
    return fig

##################################################
def generate_figure(ax, accuracy, detected_drifts, stream, detector):
    num_drifts = []
    for a, d in zip(accuracy, detected_drifts):
        num_drifts.append(len(d))
        u, v,_ = zip(*a)
        ax.step(u, v, color="blue", alpha=0.3)
        if d:
            u, v = zip(*d)
            ax.scatter(u, v, color="red", label="Detected drifts", s=10)

    # visualize true drift points (both gradual and sudden)
    for drift in stream.drift_points:
        if isinstance(drift, int):
            # sudden drift: just a vertical line at the drift time
            ax.axvline(drift, color="gray", linewidth=1.5, alpha=1.0)
        else:
            # gradual drift: (start, end)
            start_drift, end_drift = drift
            ax.axvspan(start_drift, end_drift, color="gray", alpha=0.2)
            ax.axvline(start_drift, color="gray", linewidth=1.5, alpha=1.0)
            ax.axvline(end_drift, color="gray", linewidth=1.5, alpha=1.0)


    message = "Detected drifts: "
    message += f"{np.mean(num_drifts):.2f} ± {np.std(num_drifts):.2f}"
    ax.set_title(message)
    ax.set_xlabel(detector.message())
    ###################################
##########################################
def build_true_drift_mask(max_t, drift_points, tol=0):
    """
    y_true[t] = 1 if time t is considered 'drift', else 0.

    - sudden drift at d: mark [d-tol, d+tol]
    - gradual drift (s,e): mark [s, e+tol]
    """
    y_true = np.zeros(max_t + 1, dtype=np.int8)  # index by time; ignore 0

    for d in drift_points:
        if isinstance(d, int):
            lo = abs(d - tol)
            hi = min(max_t, d + tol)
            y_true[lo : hi + 1] = 1
        else:
            s, e = d
            lo = s
            hi = min(max_t, e + tol)
            y_true[lo : hi + 1] = 1

    return y_true


def build_pred_alarm_mask(max_t, detected_drifts, alarm_window=0):
    """
    y_pred[t] = 1 if the detector fired at/around time t, else 0.

    detected_drifts: list of (t_det, acc_at_t)
    alarm_window: if >0, smear each alarm into [t_det-alarm_window, t_det+alarm_window]
    """
    y_pred = np.zeros(max_t + 1, dtype=np.int8)

    for t_det, _ in detected_drifts:
        lo = max(1, t_det - alarm_window)
        hi = min(max_t, t_det + alarm_window)
        y_pred[lo : hi + 1] = 1

    return y_pred


def confusion_counts(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, fn, tn


def precision_recall_specificity(tp, fp, fn, tn):
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return float(precision), float(recall), float(specificity)

# Mean + std across seeds/runs
def mean_std(xs):
        xs = np.array(xs, dtype=float)
        return float(xs.mean()), float(xs.std(ddof=1)) if len(xs) > 1 else 0.0

##################################################
def summarize_runs(
    accuracies,
    detected_drifts_list,
    drift_points=None,   # optional
    tol=0,
    max_t=None,
    alarm_window=0,
):
    # Always computed
    avg_acc = float(np.mean([np.mean([a for _, a,_ in acc]) for acc in accuracies]))
    avg_nr = float(np.mean([len(d) for d in detected_drifts_list]))

    alarm_counts = [len(d) for d in detected_drifts_list]
    alarm_mean, alarm_std = mean_std(alarm_counts)

    first_alarm_times = []
    for d in detected_drifts_list:
        if len(d) > 0:
            first_alarm_times.append(d[0][0])
    if first_alarm_times:
        first_alarm_mean, first_alarm_std = mean_std(first_alarm_times)
    else:
        first_alarm_mean, first_alarm_std = 0.0, 0.0

    out = {
        "AA": avg_acc,
        "ANR": avg_nr,
        "alarms_mean": alarm_mean,
        "alarms_std": alarm_std,
        "first_alarm_t_mean": first_alarm_mean,
        "first_alarm_t_std": first_alarm_std,
        "has_ground_truth_drifts": drift_points is not None,
    }

    # If no ground truth drift list, stop here
    if drift_points is None:
        return out

    # If ground truth exists, compute confusion & PR metrics
    if max_t is None:
        raise ValueError("max_t must be provided when drift_points is provided")

    y_true = build_true_drift_mask(max_t=max_t, drift_points=drift_points, tol=tol)

    precisions, recalls, specificities = [], [], []
    tps, fps, fns, tns = [], [], [], []

    for detected_drifts in detected_drifts_list:
        y_pred = build_pred_alarm_mask(
            max_t=max_t,
            detected_drifts=detected_drifts,
            alarm_window=alarm_window,
        )
        tp, fp, fn, tn = confusion_counts(y_true, y_pred)
        p, r, s = precision_recall_specificity(tp, fp, fn, tn)

        tps.append(tp); fps.append(fp); fns.append(fn); tns.append(tn)
        precisions.append(p); recalls.append(r); specificities.append(s)

    prec_m, prec_s = mean_std(precisions)
    rec_m, rec_s = mean_std(recalls)
    spec_m, spec_s = mean_std(specificities)

    tp_m, tp_s = mean_std(tps)
    fp_m, fp_s = mean_std(fps)
    fn_m, fn_s = mean_std(fns)
    tn_m, tn_s = mean_std(tns)

    out.update({
        "TP_mean": tp_m, "TP_std": tp_s,
        "FP_mean": fp_m, "FP_std": fp_s,
        "FN_mean": fn_m, "FN_std": fn_s,
        "TN_mean": tn_m, "TN_std": tn_s,
        "precision_mean": prec_m, "precision_std": prec_s,
        "recall_mean": rec_m, "recall_std": rec_s,
        "specificity_mean": spec_m, "specificity_std": spec_s,
        "tol": tol,
        "alarm_window": alarm_window,
    })
    return out

##################################################
def experiment(n, nu, gamma, max_t, stream, detector, model):
    """
    Same logic as the original notebook, but reusing ht2d utilities.
    model is an htdt classifier (e.g. HoeffdingTreeClassifier).
    """

    orig_model = copy.deepcopy(model)

    x_window = deque(maxlen=n)
    y_window = deque(maxlen=n)
    outlier_window = deque(maxlen=n)
    detected_drifts = []
    accuracy = []
    is_drift = False

    # Initial window
    accumulate_examples(stream, n, x_window, y_window)
    t = n

    # Train classifier on initial window
    model.learn_many(np.array(x_window), np.array(y_window))
    accuracy.append((n, model.window_accuracy(x_window, y_window),is_drift))

    # Train OCC on initial window
    occ = OneClassSVM(kernel="rbf", gamma=gamma, nu=nu)
    occ.fit(np.array(x_window))

    for x in x_window:
        curr_x = np.array([x[i] for i in range(len(x))])
        outlier = occ.predict(np.array([curr_x]))[0]
        outlier_window.append(outlier == -1)

    detector.reset(outlier_window)

    # Streaming loop
    for x, y in it.islice(stream.sample(), max_t - n):


        # Classifier prediction (not used for decision, only accuracy tracking)
        y_pred = model.predict_one(x)

        curr_x = to_array(x)
        x_window.append(curr_x)
        y_window.append(y)

        outlier = occ.predict(np.array([curr_x]))[0]
        outlier_window.append(outlier == -1)

        if detector.detect(outlier_window):
            accumulate_examples(stream, n, x_window, y_window, keep_fraction=keep_fraction)

            #detected_drifts.append((t, accuracy[-1][1]))
            # t += int(n * (1 - keep_fraction))
            is_drift = True

             
            # retrain model on current window
            model = copy.deepcopy(orig_model)
            model = model.__class__()  # fresh instance
            model.learn_many(np.array(x_window), np.array(y_window))

            # retrain OCC
            occ.fit(np.array(x_window))

            outlier_window.clear()
            for x in x_window:
                curr_x = np.array([x[i] for i in range(len(x))])
                outlier = occ.predict(np.array([curr_x]))[0]
                outlier_window.append(outlier == -1)
            detector.reset(outlier_window)

            a = model.window_accuracy(x_window, y_window)
            detected_drifts.append((t, a))
            accuracy.append((t, a,is_drift))
            t += int(n * (1 - keep_fraction))
        else:
            is_drift = False
            accuracy.append((t, model.window_accuracy(x_window, y_window),is_drift))
            t += 1

        if t > max_t:
            break
    final_acc = accuracy[-1][1] if accuracy else 0.0
    number_of_retrains = len(detected_drifts)

    return accuracy, detected_drifts,final_acc,number_of_retrains


In [ ]:
def build_modules(f=0.1):
    detectors = {
        #"baseline": NoDriftDetector(),
        "threshold": ThresholdDriftDetector(f=f),
        "ztest": ZTestDriftDetector(alpha=0.025),
    }

    models = {
        "ht": HoeffdingTreeClassifier(),
        "gnb": GaussianNBClassifier(),
        "pa": PAClassifier(),
        "arf": AdaptiveRandomForestClassifier(),
        "lr": LogisticRegressionClassifier(),
        #"mlp": SklearnMLPClassifier(),
        "linsvc": SklearnLinearSVCClassifier(),
        "rbfsvc": SklearnRBFSVCClassifier(),
    }

    return detectors, models


In [ ]:
def run_all_modules_on_dataset(
    X, y,
    dataset_name: str,
    out_jsonl: str,
    drift_points=None,            # optional ground truth
    seeds=(42,52,62),
    n=250,
    f=0.1,
    nu=0.1,
    gamma=0.5,
    max_t=None,
    tol=0,
    alarm_window=0,
    also_write_aggregate_summary=True,
):
    if max_t is None:
        max_t = len(y) - 1
    else:
        max_t = min(int(max_t), len(y) - 1)

    detectors, models = build_modules(f=f)

    for det_name, det in detectors.items():
        for model_name, model in models.items():

            accuracies = []
            detected = []
            

            for seed in seeds:
                stream_obj = ArrayStream(X, y, drift_points=drift_points)

                det_run = copy.deepcopy(det)
                model_run = copy.deepcopy(model)

                acc, drifts,final_acc,number_of_retrains = experiment(
                    n=n,
                    nu=nu,
                    gamma=gamma,
                    max_t=max_t,
                    stream=stream_obj,
                    detector=det_run,
                    model=model_run,
                )

                fig = generate_figure_all(
                    accuracies=[acc],              
                    detected_drifts_list=[drifts],
                    drift_points=drift_points,           # None is fine if real dataset
                    title=f"{dataset_name} | {det_name}+{model_name} | seed={seed}"
                )

                # ----------- SAVE FIGURE -----------

                plot_dir = Path("plots") / dataset_name / det_name / model_name
                plot_dir.mkdir(parents=True, exist_ok=True)

                plot_path = plot_dir / f"seed_{seed}.png"

                fig.savefig(plot_path, dpi=150, bbox_inches="tight")
                plt.close(fig)   

                accuracies.append(acc)
                detected.append(drifts)
                record = {
                    "level": "seed_run",
                    "dataset": dataset_name,
                    "detector": det_name,
                    "model": model_name,
                    "seed": int(seed),
                    "params": {
                        "n": n, "f": f, "nu": nu, "gamma": gamma,
                        "max_t": max_t, "tol": tol, "alarm_window": alarm_window,
                    },
                    "final_accuracy": final_acc,
                    "number_of_retrains": number_of_retrains,
                    "has_ground_truth_drifts": drift_points is not None,
                    "raw": {
                        "accuracy_curve": acc,         
                        "detected_drifts": drifts,       
                    },
                }

                append_jsonl(out_jsonl, record)
                print(f"[saved seed] {dataset_name} | {det_name} + {model_name} | seed={seed}")

            if also_write_aggregate_summary:
                summary = summarize_runs(
                    accuracies=accuracies,
                    detected_drifts_list=detected,
                    drift_points=drift_points,
                    tol=tol,
                    max_t=max_t if drift_points is not None else None,
                    alarm_window=alarm_window,
                )

                summary_record  = {
                    "level": "aggregate",
                    "dataset": dataset_name,
                    "detector": det_name,
                    "model": model_name,
                    "params": {
                        "n": n, "f": f, "nu": nu, "gamma": gamma,
                        "max_t": max_t, "tol": tol, "alarm_window": alarm_window,
                        "seeds": list(map(int, seeds)),
                    },
                    "summary": summary,
                }

                append_jsonl(out_jsonl, summary_record)
                print(f"[saved agg]  {dataset_name} | {det_name} + {model_name} -> {summary}")


In [ ]:
csv_path = "./data/real-world/elec.csv"   # dataset
#csv_path = "./data/artificial/rotatingHyperplane.csv"   # dataset
dataset_name = Path(csv_path).stem

X, y, feat_names = load_xy_from_csv(
    csv_path,
    target_col="target",
    standardize=False, 
)

# ---- SHUFFLE HERE ----
import numpy as np
rng = np.random.default_rng(42)
idx = rng.permutation(len(y))
X = X[idx]
y = y[idx]
# ----------------------

print("dataset:", dataset_name)
print("X shape:", X.shape, "y shape:", y.shape)
print("first features:", feat_names[:10])
print("unique y:", np.unique(y)[:20])


In [ ]:
run_all_modules_on_dataset(
    X, y,
    dataset_name=dataset_name,
    out_jsonl=f"runs/{dataset_name}.jsonl",
    drift_points=None,      # <- no GT drifts
    seeds=[42],
    n=250,
    f=0.1,
    nu=0.1,
    gamma=0.5,
    max_t=len(y)-1,
)
  

Graphs

In [ ]:

def _read_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def _curve_ts_acc(rec):
    """
    Supports raw.accuracy_curve items that are (t, acc) OR (t, acc, extra...).
    Returns ts(list[int]), acc(list[float]).
    """
    raw = rec.get("raw", {})
    curve = raw.get("accuracy_curve", []) or []
    ts, accs = [], []
    for item in curve:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            ts.append(int(item[0]))
            accs.append(float(item[1]))
    return ts, accs


def _acc_at_t(ts, accs, t):
    """
    Get accuracy at drift time t:
    - exact match if t exists in ts
    - otherwise nearest neighbor in time
    """
    if not ts:
        return None
    t = int(t)
    # exact
    try:
        idx = ts.index(t)
        return float(accs[idx])
    except ValueError:
        pass
    # nearest
    arr = np.asarray(ts, dtype=int)
    idx = int(np.argmin(np.abs(arr - t)))
    return float(accs[idx])

def _det_points_from_record(rec):
    """
    Return list of (t_det, acc_at_t) where acc_at_t is taken from the accuracy curve.
    This guarantees dots lie exactly on the plotted curve.
    """
    raw = rec.get("raw", {})
    det = raw.get("detected_drifts", []) or []

    drift_ts = []
    for d in det:
        if isinstance(d, (list, tuple)) and len(d) >= 1:
            drift_ts.append(int(d[0]))

    ts, accs = _curve_ts_acc(rec)

    det_points = []
    for t in drift_ts:
        a = _acc_at_t(ts, accs, t)
        if a is not None:
            det_points.append((int(t), float(a)))

    return det_points


def _mean_std(lst):
    if not lst:
        return 0.0, 0.0
    m = float(np.mean(lst))
    s = float(np.std(lst, ddof=1)) if len(lst) > 1 else 0.0
    return m, s


def generate_compare_figure(
    ztest_runs, threshold_runs,
    ztest_det_points_runs, threshold_det_points_runs,
    title=None,
    figsize=(8, 3),
):
    fig, ax = plt.subplots(figsize=figsize)

    z_counts = [len(d) for d in ztest_det_points_runs]
    t_counts = [len(d) for d in threshold_det_points_runs]
    z_m, z_s = _mean_std(z_counts)
    t_m, t_s = _mean_std(t_counts)

    # ---- Z-TEST CURVES ----
    first = True
    for run_curve in ztest_runs:
        if not run_curve:
            continue
        t_vals, acc_vals = zip(*run_curve)
        if first:
            ax.plot(
                t_vals, acc_vals,
                color="red",
                alpha=0.3,
                label="Z-Test",
                zorder=1
            )
            first = False
        else:
            ax.plot(t_vals, acc_vals, color="red", alpha=0.3)

    # ---- THRESHOLD CURVES ----
    first = True
    for run_curve in threshold_runs:
        if not run_curve:
            continue
        t_vals, acc_vals = zip(*run_curve)
        if first:
            ax.plot(
                t_vals, acc_vals,
                color="blue",
                alpha=0.3,
                label="Threshold",
                zorder=1
            )
            first = False
        else:
            ax.plot(t_vals, acc_vals, color="blue", alpha=0.3)

    # ---- Z-TEST DRIFT DOTS ----
    first = True
    for det_points in ztest_det_points_runs:
        if not det_points:
            continue
        t_det, acc_det = zip(*det_points)
        if first:
            ax.scatter(
                t_det, acc_det,
                color="red",
                s=12,
                label="Z-Test - Drift Points",
                zorder=10
            )
            first = False
        else:
            ax.scatter(t_det, acc_det, color="red", s=12)

    # ---- THRESHOLD DRIFT DOTS ----
    first = True
    for det_points in threshold_det_points_runs:
        if not det_points:
            continue
        t_det, acc_det = zip(*det_points)
        if first:
            ax.scatter(
                t_det, acc_det,
                color="blue",
                s=12,
                label="Threshold - Drift Points",
                zorder=10
            )
            first = False
        else:
            ax.scatter(t_det, acc_det, color="blue", s=12)

    msg = (
        f"Z-Test drifts: {z_m:.2f} ± {z_s:.2f} (n={len(z_counts)}) | "
        f"Threshold drifts: {t_m:.2f} ± {t_s:.2f} (n={len(t_counts)})"
    )

    # ax.set_xlabel("")
    # ax.set_ylabel("")
    ax.tick_params(axis='both', labelsize=16)
    ax.set_xlim(left=0)
    #ax.set_xlim(0,100000)
    ax.set_ylim(0.7, 1.0)  
    ax.grid(False)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    #ax.legend()

    return fig


def generate_compare_plots_threshold_vs_ztest(
    jsonl_path: str,
    dataset_name: str,
    out_root: str = "plots_compare",
    model_filter=None,              # None or list like ["ht","gnb"]
    level: str = "seed_run",
    det_z: str = "ztest",
    det_t: str = "threshold",
    figsize=(8, 3),
    dpi: int = 150,
):
    """
    Creates one plot per model,
    but comparing ztest vs threshold. Saves:
      plots_compare/<dataset>/<model>/ztest_vs_threshold.png

    """
    # Group records by (model, detector)
    recs = defaultdict(list)
    for r in _read_jsonl(jsonl_path):
        if r.get("level") != level:
            continue
        if r.get("dataset") != dataset_name:
            continue
        if r.get("detector") not in (det_z, det_t):
            continue
        if "raw" not in r:
            continue
        recs[(r.get("model"), r.get("detector"))].append(r)

    models = sorted({m for (m, d) in recs.keys()})
    if model_filter is not None:
        models = [m for m in models if m in set(model_filter)]

    saved = []

    for model in models:
        z_recs = recs.get((model, det_z), [])
        t_recs = recs.get((model, det_t), [])
        if not z_recs or not t_recs:
            continue

        # Per-run curves and per-run drift-point coords (t_det, acc_det)
        z_runs, z_det_points_runs = [], []
        for r in z_recs:
            ts, accs = _curve_ts_acc(r)
            z_runs.append(list(zip(ts, accs)))
            z_det_points_runs.append(_det_points_from_record(r))

        t_runs, t_det_points_runs = [], []
        for r in t_recs:
            ts, accs = _curve_ts_acc(r)
            t_runs.append(list(zip(ts, accs)))
            t_det_points_runs.append(_det_points_from_record(r))

        # Generate figure (style like your generate_figure_all)
        title = f"{dataset_name} | {det_t}+{model} vs {det_z}+{model}"
        fig = generate_compare_figure(
            ztest_runs=z_runs,
            threshold_runs=t_runs,
            ztest_det_points_runs=z_det_points_runs,
            threshold_det_points_runs=t_det_points_runs,
            title=title,
            figsize=figsize,
        )

        out_dir = Path(out_root) / dataset_name / model
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{det_z}_vs_{det_t}.png"

        fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)

        saved.append(str(out_path))

    return saved

In [ ]:
data_path = "phishing"  
dataset_name = "phishing"  
jsonl_path = f"runs/{data_path}.jsonl"

saved = generate_compare_plots_threshold_vs_ztest(
    jsonl_path=jsonl_path,
    dataset_name=dataset_name,
    out_root="plots_compare_final",
    det_z="ztest",
    det_t="threshold",
    level="seed_run",
    figsize=(14, 3), 
)

print("Saved:", len(saved))
for p in saved:
    print(p)

from pathlib import Path
print("Compare plots folder:", Path("plots_compare").resolve())

Results and Ratios

In [ ]:

dataset_name = "airlines" 
jsonl_path = f"runs/{dataset_name}.jsonl"
 

# ---------------------------
# Read JSONL
# ---------------------------
records = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

# ---------------------------
# Separate seed_runs and aggregates
# ---------------------------
seed_runs = defaultdict(list)   # key: (model, detector) -> list[record]
aggregates = {}                 # key: (model, detector) -> summary dict

for r in records:
    model = r.get("model")
    detector = r.get("detector")
    level = r.get("level")
    if model is None or detector is None or level is None:
        continue

    key = (model, detector)

    if level == "seed_run":
        seed_runs[key].append(r)
    elif level == "aggregate":
        aggregates[key] = r.get("summary", {}) or {}

# ---------------------------
# Compute stats for each (model, detector)
# ---------------------------
stats = {}
for (model, detector), runs in seed_runs.items():
    all_acc_vals = []
    retrain_counts = []
    drifts_flat = []

    for r in sorted(runs, key=lambda rr: rr.get("seed", 0)):
        raw = r.get("raw", {}) or {}

        curve = raw.get("accuracy_curve", []) or []
        for item in curve:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                all_acc_vals.append(float(item[1]))

        det = raw.get("detected_drifts", []) or []
        retrain_counts.append(len(det))
        drifts_flat.extend(det)

    avg_acc = float(np.mean(all_acc_vals)) if all_acc_vals else np.nan
    avg_retrains = float(np.mean(retrain_counts)) if retrain_counts else np.nan

    summary = aggregates.get((model, detector), {}) or {}
    AA = summary.get("AA", None)
    ANR = summary.get("ANR", None)

    stats[(model, detector)] = {
        "runs": len(runs),
        "avg_acc": avg_acc,
        "avg_retrains": avg_retrains,
        "AA": AA,
        "ANR": ANR,
        "first10_drifts": drifts_flat[:10],
    }

# ---------------------------
# Print per model: both detectors + ratios
# ---------------------------
models = sorted({m for (m, _) in stats.keys()})

for model in models:
    z = stats.get((model, "ztest"))
    t = stats.get((model, "threshold"))

    # If one detector missing, still print what exists
    print(f"\n====================  MODEL: {model}  ====================")

    if z:
        print("\n[ZTEST]")
        print(f"  seed_runs: {z['runs']}")
        print(f"  Avg Accuracy (from curve): {z['avg_acc']:.6f}" if np.isfinite(z["avg_acc"]) else "  Avg Accuracy: N/A")
        print(f"  Avg # Retrains: {z['avg_retrains']:.2f}" if np.isfinite(z["avg_retrains"]) else "  Avg # Retrains: N/A")
        print(f"  AA (from summary): {z['AA']}")
        print(f"  ANR (from summary): {z['ANR']}")
        print(f"  First 10 detected_drifts: {z['first10_drifts']}")
    else:
        print("\n[ZTEST]  (missing)")

    if t:
        print("\n[THRESHOLD]")
        print(f"  seed_runs: {t['runs']}")
        print(f"  Avg Accuracy (from curve): {t['avg_acc']:.6f}" if np.isfinite(t["avg_acc"]) else "  Avg Accuracy: N/A")
        print(f"  Avg # Retrains: {t['avg_retrains']:.2f}" if np.isfinite(t["avg_retrains"]) else "  Avg # Retrains: N/A")
        print(f"  AA (from summary): {t['AA']}")
        print(f"  ANR (from summary): {t['ANR']}")
        print(f"  First 10 detected_drifts: {t['first10_drifts']}")
    else:
        print("\n[THRESHOLD]  (missing)")

    # Ratios (only if both exist and denominators are valid)
    print("\n[RATIOS]")
    if z and t and np.isfinite(z["avg_acc"]) and np.isfinite(t["avg_acc"]) and t["avg_acc"] != 0:
        p_acc = z["avg_acc"] / t["avg_acc"]
        print(f"  p_acc = AvgAcc(ztest) / AvgAcc(threshold) = {p_acc:.4f}")
    else:
        print("  p_acc = N/A")

    if z and t and np.isfinite(z["avg_retrains"]) and np.isfinite(t["avg_retrains"]) and z["avg_retrains"] != 0:
        p_retrain = t["avg_retrains"] / z["avg_retrains"]
        print(f"  p_retrain = AvgRetr(threshold) / AvgRetr(ztest) = {p_retrain:.4f}")
    else:
        print("  p_retrain = N/A")